SOURCE TO TARGET (ROW COUNT VALIDATION)

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA bronze;
SELECT COUNT(*) AS bronze_count
FROM bronze.customers_raw;

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA silver;
SELECT COUNT(*) AS silver_count
FROM silver.customers_clean;

DATA QUALITY VALIDATION

In [0]:
%sql
-- DUPLICATE CHECK (Bronze)

USE CATALOG retail_sales_catalog;
USE SCHEMA bronze;
SELECT CustomerID, COUNT(*) AS cnt
FROM bronze.customers_raw
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:
%sql
-- DUPLICATE CHECK (Silver)

USE CATALOG retail_sales_catalog;
USE SCHEMA silver;
SELECT CustomerID, COUNT(*)
FROM silver.customers_clean
GROUP BY CustomerID
HAVING COUNT(*) > 1;

SCD TYPE 2 VALIDATION

In [0]:
%sql
SELECT CustomerID, COUNT(*)
FROM gold.dim_customer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING COUNT(*) > 1;

FACT TABLE VALIDATION

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA gold;
SELECT COUNT(*) FROM gold.fact_sales;

ARCHIVAL VALIDATION

In [0]:
files = dbutils.fs.ls("s3://retaill-client/retaill-lakehouse/landing/")

customers_files = [f.name for f in files if "customers" in f.name]

print(customers_files)
print("Count:", len(customers_files))

In [0]:
for entity in ["customers", "products", "stores", "sales"]:
    entity_files = [f.name for f in files if entity in f.name]
    
    if len(entity_files) > 1:
        print(f"❌ Archival failed for {entity}")
    else:
        print(f"✔ {entity} is correct")

Check SCD:

In [0]:
%sql
SELECT *
FROM gold.dim_customer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM silver.customers_clean
)
ORDER BY CustomerID, StartDate;

Check Fact:

In [0]:
%sql
SELECT COUNT(*) FROM gold.fact_sales;

Check Archive:

In [0]:
display(dbutils.fs.ls("s3://retaill-client/retaill-lakehouse/archive/"))